# SymPy — линейная алгебра с LaTeX вводом/выводом

Вставляй матрицу в формате LaTeX и получай результат. Сначала запусти ячейку **Setup**.

## ⚙️ Setup — запускать первым

In [2]:
from sympy import *
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application
from IPython.display import display, Math
import re

_transformations = standard_transformations + (implicit_multiplication_application,)

def parse_latex_matrix(s: str) -> Matrix:
    """Парсит LaTeX-матрицу \\begin{pmatrix/bmatrix/matrix}...\\end{...} → SymPy Matrix."""
    s = s.strip()
    match = re.search(
        r'\\begin\{[a-z]*matrix\*?\}(.*?)\\end\{[a-z]*matrix\*?\}',
        s, re.DOTALL
    )
    if not match:
        raise ValueError(
            "Не найдена матрица. Используй \\begin{pmatrix} ... \\end{pmatrix}"
        )
    content = match.group(1).strip()
    rows = re.split(r'\\\\', content)
    data = []
    for row in rows:
        row = row.strip()
        if not row:
            continue
        cols = [
            parse_expr(c.strip(), transformations=_transformations)
            for c in row.split('&')
        ]
        data.append(cols)
    return Matrix(data)

def _show(latex_str: str):
    display(Math(latex_str))

print("✓ SymPy", __import__('sympy').__version__, "— парсер готов")

✓ SymPy 1.13.1 — парсер готов


---
## 1. Умножение матриц — $\mathbf{A} \cdot \mathbf{B} = \mathbf{C}$

In [3]:
# ── Умножение матриц ─────────────────────────────────────────────────────────
# Замени матрицы ниже на свои (pmatrix / bmatrix — оба работают)

A_latex = r"""
\begin{pmatrix}
1 & 2 \\
3 & 4
\end{pmatrix}
"""

B_latex = r"""
\begin{pmatrix}
5 & 6 \\
7 & 8
\end{pmatrix}
"""

# ─────────────────────────────────────────────────────────────────────────────
A = parse_latex_matrix(A_latex)
B = parse_latex_matrix(B_latex)
C = A * B

_show(
    r"\mathbf{A} \cdot \mathbf{B} "
    r"= " + latex(A) +
    r"\cdot" + latex(B) +
    r"= " + latex(C)
)
latex(C)

<IPython.core.display.Math object>

'\\left[\\begin{matrix}19 & 22\\\\43 & 50\\end{matrix}\\right]'

---
## 2. Определитель — $\det(\mathbf{A}) = n$

In [4]:
# ── Определитель ─────────────────────────────────────────────────────────────
# Только квадратная матрица!

A_latex = r"""
\begin{pmatrix}
1 & 2 & 3 \\
0 & 4 & 5 \\
1 & 0 & 6
\end{pmatrix}
"""

# ─────────────────────────────────────────────────────────────────────────────
A = parse_latex_matrix(A_latex)
d = A.det()

_show(r"\det " + latex(A) + r" = " + latex(d))

<IPython.core.display.Math object>

---
## 3. Метод Гаусса — прямой ход по шагам

Вводи расширенную матрицу системы $[A \mid b]$ (последний столбец — правая часть).  
Можно вводить и просто матрицу без правой части.

In [5]:
# ── Метод Гаусса (прямой ход по шагам) ──────────────────────────────────────
# Расширенная матрица [A|b]: последний столбец — правая часть

M_latex = r"""
\begin{pmatrix}
2  &  1 & -1 &  8 \\
-3 & -1 &  2 & -11 \\
-2 &  1 &  2 & -3
\end{pmatrix}
"""

# ─────────────────────────────────────────────────────────────────────────────
M = parse_latex_matrix(M_latex).applyfunc(Rational)  # точная арифметика
m, n = M.shape

steps   = [M.copy()]
captions = [r"\text{Исходная матрица}"]

pivot_row = 0
for col in range(n):
    # ищем ненулевой элемент в текущем столбце
    found = next((r for r in range(pivot_row, m) if M[r, col] != 0), None)
    if found is None:
        continue

    # перестановка строк
    if found != pivot_row:
        M.row_swap(found, pivot_row)
        steps.append(M.copy())
        captions.append(
            fr"R_{{{pivot_row+1}}} \leftrightarrow R_{{{found+1}}}"
        )

    # обнуляем элементы ниже ведущего
    for row in range(pivot_row + 1, m):
        if M[row, col] == 0:
            continue
        factor = M[row, col] / M[pivot_row, col]
        for j in range(n):
            M[row, j] = M[row, j] - factor * M[pivot_row, j]
        f_str = latex(factor)
        steps.append(M.copy())
        captions.append(
            fr"R_{{{row+1}}} \leftarrow "
            fr"R_{{{row+1}}} - {f_str} \cdot R_{{{pivot_row+1}}}"
        )

    pivot_row += 1

# --- Вывод всех шагов ---
for i, (cap, step) in enumerate(zip(captions, steps)):
    _show(cap)
    _show(latex(step))
    if i < len(steps) - 1:
        print()  # небольшой отступ между шагами

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

---
## 4. Проверка линейной независимости (ЛНЗ)

Каждая **строка** матрицы — один вектор.

In [6]:
# ── Линейная независимость (ЛНЗ) ─────────────────────────────────────────────
# Каждая СТРОКА — один вектор; система ЛНЗ ⟺ rank = число строк

A_latex = r"""
\begin{pmatrix}
1 &  0 &  2 \\
0 &  1 & -1 \\
3 & -1 &  7
\end{pmatrix}
"""

# ─────────────────────────────────────────────────────────────────────────────
A = parse_latex_matrix(A_latex)
k   = A.rows           # число векторов
rk  = A.rank()

_show(r"A = " + latex(A))
_show(r"\mathrm{rank}(A) = " + latex(rk) + r",\quad k = " + str(k))

if rk == k:
    _show(r"\Rightarrow \text{Векторы \textbf{линейно независимы} (ЛНЗ)}")
else:
    deficit = k - rk
    _show(
        r"\Rightarrow \text{Векторы \textbf{линейно зависимы}: }"
        r"\mathrm{rank} = " + latex(rk) +
        r" < k = " + str(k) +
        r",\ \text{дефект} = " + str(deficit)
    )
    # показываем нетривиальную линейную комбинацию
    ns = A.T.nullspace()   # нуль-пространство A^T → коэффициенты зависимости
    if ns:
        c = ns[0]
        terms = " + ".join(
            fr"{latex(c[i])} \mathbf{{v}}_{{{i+1}}}"
            for i in range(k)
        )
        _show(r"\text{Нетривиальная ЛК: } " + terms + r" = \mathbf{0}")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

---
## 5. ФСР — фундаментальная система решений $A\mathbf{x} = \mathbf{0}$

Вводи матрицу $A$ (без правой части). ФСР — базис ядра $\ker A$.

In [7]:
# ── ФСР (ker A, нуль-пространство) ───────────────────────────────────────────
# Матрица A системы Ax = 0

A_latex = r"""
\begin{pmatrix}
1 &  2 & -1 &  3 \\
2 &  4 &  0 &  2 \\
3 &  6 &  1 &  1
\end{pmatrix}
"""

# ─────────────────────────────────────────────────────────────────────────────
A = parse_latex_matrix(A_latex)
n   = A.cols
rk  = A.rank()
dim = n - rk   # размерность ядра = дефект

_show(r"A = " + latex(A))
_show(
    r"\mathrm{rank}(A) = " + str(rk) +
    r",\quad n = " + str(n) +
    r",\quad \dim\ker A = n - \mathrm{rank} = " + str(dim)
)

ns = A.nullspace()   # список базисных векторов ядра (ФСР)

if not ns:
    _show(r"\ker A = \{\mathbf{0}\} \quad \text{— ФСР пуста, только нулевое решение}")
else:
    _show(r"\text{ФСР содержит } " + str(len(ns)) + r" \text{ вектор(а/ов):}")
    fsr_terms = r",\quad ".join(
        r"\mathbf{v}_{" + str(i+1) + r"} = " + latex(v)
        for i, v in enumerate(ns)
    )
    _show(fsr_terms)

    # Общее решение
    free_vars = [Symbol(f"c_{{{i+1}}}") for i in range(len(ns))]
    x_general = zeros(n, 1)
    for i in range(len(ns)):
        x_general = x_general + free_vars[i] * ns[i]
    coef_str  = r",\ ".join(latex(c) for c in free_vars) + r"\ \in \mathbb{R}"
    _show(
        r"\mathbf{x} = " +
        " + ".join(
            latex(free_vars[i]) + r"\cdot " + latex(ns[i])
            for i in range(len(ns))
        ) +
        r"\quad (" + coef_str + r")"
    )


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>